# imporve model
- K-Fold (StratifiedKFold) 사용
- XGB + RandomForest → Stacking
- 최종 모델: LogisticRegression

XGB + RF → 결과 → LogisticRegression이 다시 학습

In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
# 데이터 로드
train_df = pd.read_csv("data_train.csv")
test_df = pd.read_csv("data_test.csv")

In [3]:
# 불필요한 칼럼제거
# null값이 반 이상
drop_cols = ["insolation time(hr)", "insolation(MJ/m2)"]
train_df = train_df.drop(columns=drop_cols)
test_df = test_df.drop(columns=drop_cols)


In [4]:
# 특성/라벨 분리

X = train_df.drop(columns=["weather label"])
y = train_df["weather label"]

In [5]:
# 결측치 처리
imputer = SimpleImputer(strategy="median") # 중앙값
X_cols = X.columns
X = pd.DataFrame(imputer.fit_transform(X), columns=X_cols)
test_df = pd.DataFrame(imputer.transform(test_df), columns=X_cols)


In [6]:
# feature engineering
def apply_feature_engineering(df):
    df['temp_dew_diff'] = df['temperature(C)'] - df['dew point(C)']
    df['pressure_diff'] = df['Sea level pressure(hPa)'] - df['atmospheric pressure(hPa)']

    df["wind_dir_sin"] = np.sin(np.radians(df["wind direction"]))
    df["wind_dir_cos"] = np.cos(np.radians(df["wind direction"]))

    df['wind_u'] = df['wind speed(m/s)'] * df['wind_dir_cos']
    df['wind_v'] = df['wind speed(m/s)'] * df['wind_dir_sin']

    df['apparent_temp'] = df['temperature(C)'] * 0.7 + df['humidity(%)'] * 0.01

    df['THI'] = 1.8 * df['temperature(C)'] - 0.55 * (1 - df['humidity(%)']/100) * (1.8 * df['temperature(C)'] - 26) + 32

    df = df.drop(columns=["wind direction"])
    return df

In [7]:
# 적용
X = apply_feature_engineering(X)
X_test = apply_feature_engineering(test_df)

print("특성 개수:", len(X.columns))
X.head()

특성 개수: 16


,temperature(C),wind speed(m/s),humidity(%),vapor pressure(hPa),dew point(C),atmospheric pressure(hPa),Sea level pressure(hPa),cloud,temp_dew_diff,pressure_diff,wind_dir_sin,wind_dir_cos,wind_u,wind_v,apparent_temp,THI
0,23.1,1.3,84.0,23.7,20.2,999.3,1005.5,10.0,2.9,6.2,1.000000,6.123234e-17,7.960204e-17,1.300000,17.01,72.20896
1,25.5,1.1,63.0,20.5,17.9,1017.3,1018.8,4.0,7.6,1.5,0.939693,3.420201e-01,3.762222e-01,1.033662,18.48,73.85035
2,4.2,2.1,88.0,7.2,2.3,1002.9,1030.4,8.0,1.9,27.5,-0.939693,-3.420201e-01,-7.182423e-01,-1.973355,3.82,40.77704
3,5.4,1.0,21.0,1.9,-15.2,1027.4,1030.8,6.0,20.6,3.4,1.000000,6.123234e-17,6.123234e-17,1.000000,3.99,48.79366
4,23.6,2.7,26.0,7.5,2.9,1004.1,1014.4,0.0,20.7,10.3,-0.939693,-3.420201e-01,-9.234544e-01,-2.537170,16.78,67.77264


In [8]:
# 모델 정의
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=1.0,
    reg_lambda=1.5,
    reg_alpha=0.5,
    random_state=42
)

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

estimators = [
    ('xgb', xgb_model),
    ('rf', rf_model)
]

stacking_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(),
    cv=5,
    n_jobs=-1
)

In [9]:
# K-fold 학습 + OOF
print("K-Fold 시작")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds_probs = np.zeros((len(X_test), len(np.unique(y))))

fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"Fold {fold+1}")

    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

    stacking_clf.fit(X_tr, y_tr)

    val_pred = stacking_clf.predict(X_va)
    oof_preds[val_idx] = val_pred

    score = accuracy_score(y_va, val_pred)
    fold_scores.append(score)

    print("Accuracy:", score)

    test_preds_probs += stacking_clf.predict_proba(X_test) / skf.n_splits

K-Fold 시작
Fold 1
Accuracy: 0.779126213592233
Fold 2
Accuracy: 0.7864077669902912
Fold 3
Accuracy: 0.7305825242718447
Fold 4
Accuracy: 0.7737226277372263
Fold 5
Accuracy: 0.7445255474452555


In [11]:
oof_accuracy = accuracy_score(y, oof_preds)
print("OOF Accuracy:", oof_accuracy)

final_test_preds = np.argmax(test_preds_probs, axis=1)

submission = pd.DataFrame({
    'weather label': final_test_preds
})

submission.to_csv('pred_imporve.csv', index=False)
print("저장 완료")

OOF Accuracy: 0.7628765792031098
저장 완료


# improve_model_voting
Stacking 대신 Voting 사용

XGB + RF → 그냥 합쳐서 평균 (soft voting)

In [13]:
# 라이브러리 import
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

In [15]:
# 데이터 불러오기
train_df = pd.read_csv("data_train.csv")
test_df = pd.read_csv("data_test.csv")

train_df.head()

,temperature(C),wind speed(m/s),wind direction,humidity(%),vapor pressure(hPa),dew point(C),atmospheric pressure(hPa),Sea level pressure(hPa),insolation time(hr),insolation(MJ/m2),cloud,weather label
0,23.1,1.3,90.0,84.0,23.7,20.2,999.3,1005.5,NaN,NaN,10,1
1,25.5,1.1,70.0,63.0,20.5,17.9,1017.3,1018.8,0.9,NaN,4,0
2,4.2,2.1,250.0,88.0,7.2,2.3,1002.9,1030.4,NaN,NaN,8,1
3,5.4,1.0,90.0,21.0,1.9,-15.2,1027.4,1030.8,1.0,NaN,6,3
4,23.6,2.7,250.0,26.0,7.5,2.9,1004.1,1014.4,1.0,3.39,0,0


In [16]:
# 필요 없는 컬럼 제거
drop_cols = ["insolation time(hr)", "insolation(MJ/m2)"]

train_df = train_df.drop(columns=drop_cols)
test_df = test_df.drop(columns=drop_cols)

# 입력 / 정답 분리
X = train_df.drop(columns=["weather label"])
y = train_df["weather label"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (2058, 9)
y shape: (2058,)


In [17]:
# 결측치는 중앙값으로
imputer = SimpleImputer(strategy="median")

X_cols = X.columns

# train 기준으로 학습하고
X = pd.DataFrame(imputer.fit_transform(X), columns=X_cols)

# test에도 동일하게 적용
test_df = pd.DataFrame(imputer.transform(test_df), columns=X_cols)

In [18]:
def apply_feature_engineering(df):

    # 온도 - 이슬점 차이 (습도 관련)
    df['temp_dew_diff'] = df['temperature(C)'] - df['dew point(C)']

    # 해수면 기압 - 실제 기압 (압력 차이)
    df['pressure_diff'] = df['Sea level pressure(hPa)'] - df['atmospheric pressure(hPa)']

    # 풍향을 sin/cos로 변환 (각도 → 수치)
    df["wind_dir_sin"] = np.sin(np.radians(df["wind direction"]))
    df["wind_dir_cos"] = np.cos(np.radians(df["wind direction"]))

    # 풍속 + 방향 → 벡터화
    df['wind_u'] = df['wind speed(m/s)'] * df['wind_dir_cos']
    df['wind_v'] = df['wind speed(m/s)'] * df['wind_dir_sin']

    # 체감 온도
    df['apparent_temp'] = df['temperature(C)'] * 0.7 + df['humidity(%)'] * 0.01

    # 불쾌지수 (THI)
    df['THI'] = 1.8 * df['temperature(C)'] - 0.55 * (1 - df['humidity(%)']/100) * (1.8 * df['temperature(C)'] - 26) + 32

    # 원래 wind direction 제거 (이미 sin/cos로 표현됨)
    df = df.drop(columns=["wind direction"])

    return df

In [19]:
# train / test 둘 다 동일하게 적용
X = apply_feature_engineering(X)
X_test = apply_feature_engineering(test_df)

print("최종 feature 개수:", len(X.columns))
X.head()

최종 feature 개수: 16


,temperature(C),wind speed(m/s),humidity(%),vapor pressure(hPa),dew point(C),atmospheric pressure(hPa),Sea level pressure(hPa),cloud,temp_dew_diff,pressure_diff,wind_dir_sin,wind_dir_cos,wind_u,wind_v,apparent_temp,THI
0,23.1,1.3,84.0,23.7,20.2,999.3,1005.5,10.0,2.9,6.2,1.000000,6.123234e-17,7.960204e-17,1.300000,17.01,72.20896
1,25.5,1.1,63.0,20.5,17.9,1017.3,1018.8,4.0,7.6,1.5,0.939693,3.420201e-01,3.762222e-01,1.033662,18.48,73.85035
2,4.2,2.1,88.0,7.2,2.3,1002.9,1030.4,8.0,1.9,27.5,-0.939693,-3.420201e-01,-7.182423e-01,-1.973355,3.82,40.77704
3,5.4,1.0,21.0,1.9,-15.2,1027.4,1030.8,6.0,20.6,3.4,1.000000,6.123234e-17,6.123234e-17,1.000000,3.99,48.79366
4,23.6,2.7,26.0,7.5,2.9,1004.1,1014.4,0.0,20.7,10.3,-0.939693,-3.420201e-01,-9.234544e-01,-2.537170,16.78,67.77264


In [21]:
# XGBoost 모델
xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# Random Forest 모델
rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=15,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)

# 두 모델을 합쳐서 Voting Ensemble 구성
estimators = [
    ('xgb', xgb_model),
    ('rf', rf_model)
]

# soft voting → 확률 평균
# weights=[2,1] → XGB를 더 믿는다
voting_clf = VotingClassifier(
    estimators=estimators,
    voting='soft',
    weights=[2, 1]
)

In [22]:
# StratifiedKFold → 클래스 비율 유지하면서 나눔
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# OOF 예측값 저장
oof_preds = np.zeros(len(X))

# test 예측 확률 누적
test_preds_probs = np.zeros((len(X_test), len(np.unique(y))))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"Fold {fold+1}")

    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

    voting_clf.fit(X_tr, y_tr)

    val_pred = voting_clf.predict(X_va)

    oof_preds[val_idx] = val_pred

    test_preds_probs += voting_clf.predict_proba(X_test) / skf.n_splits

Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


In [24]:
# 전체 OOF 정확도
oof_accuracy = accuracy_score(y, oof_preds)
print(f"OOF Accuracy: {oof_accuracy:.4f}")

# 확률 → 최종 클래스 선택
final_test_preds = np.argmax(test_preds_probs, axis=1)

submission = pd.DataFrame({
    'weather label': final_test_preds
})

submission.to_csv('pred_imporve_voting.csv', index=False)


OOF Accuracy: 0.7663


# advanced

모델 3개:
- XGB
- HistGradientBoosting
- CatBoost
- Soft Voting
- feature engineering 더 많이
- 결측치도 모델이 직접 처리 (imputer 없음)

XGB + HGB + CatBoost → soft voting

In [26]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.4 MB/s eta 0:00:00


In [27]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier, VotingClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score


In [28]:
train_df = pd.read_csv("data_train.csv")
test_df = pd.read_csv("data_test.csv")

train_df.head()

,temperature(C),wind speed(m/s),wind direction,humidity(%),vapor pressure(hPa),dew point(C),atmospheric pressure(hPa),Sea level pressure(hPa),insolation time(hr),insolation(MJ/m2),cloud,weather label
0,23.1,1.3,90.0,84.0,23.7,20.2,999.3,1005.5,NaN,NaN,10,1
1,25.5,1.1,70.0,63.0,20.5,17.9,1017.3,1018.8,0.9,NaN,4,0
2,4.2,2.1,250.0,88.0,7.2,2.3,1002.9,1030.4,NaN,NaN,8,1
3,5.4,1.0,90.0,21.0,1.9,-15.2,1027.4,1030.8,1.0,NaN,6,3
4,23.6,2.7,250.0,26.0,7.5,2.9,1004.1,1014.4,1.0,3.39,0,0


In [29]:
X = train_df.drop(columns=["weather label"])
y = train_df["weather label"]

X_test = test_df.copy()

print("Train shape:", X.shape)
print("Test shape:", X_test.shape)

Train shape: (2058, 11)
Test shape: (485, 11)


In [30]:
# 여기서는 결측치를 따로 채우지 않음
# XGB, HGB, CatBoost는 결측치를 "자동 처리"할 수 있기때문에

print("결측치를 그대로 두고 모델이 직접 처리하도록 설정")

결측치를 그대로 두고 모델이 직접 처리하도록 설정


In [31]:
def apply_feature_engineering(df):

    # 온도 - 이슬점 → 습도/안개 관련 정보
    df['temp_dew_diff'] = df['temperature(C)'] - df['dew point(C)']

    # 기압 차이 → 날씨 변화 신호
    df['pressure_diff'] = df['Sea level pressure(hPa)'] - df['atmospheric pressure(hPa)']

    # 풍향을 sin/cos로 변환 (각도 문제 해결)
    df["wind_dir_sin"] = np.sin(np.radians(df["wind direction"]))
    df["wind_dir_cos"] = np.cos(np.radians(df["wind direction"]))

    # 풍속 + 방향 → 벡터
    df['wind_u'] = df['wind speed(m/s)'] * df['wind_dir_cos']
    df['wind_v'] = df['wind speed(m/s)'] * df['wind_dir_sin']

    # 체감 온도
    df['apparent_temp'] = df['temperature(C)'] * 0.7 + df['humidity(%)'] * 0.01

    # 불쾌지수 (THI)
    df['THI'] = 1.8 * df['temperature(C)'] - 0.55 * (1 - df['humidity(%)']/100) * (1.8 * df['temperature(C)'] - 26) + 32

    # 추가된 interaction feature
    df['humidity_temp_ratio'] = df['humidity(%)'] / (df['temperature(C)'] + 50)
    df['pressure_temp_ratio'] = df['atmospheric pressure(hPa)'] / (df['temperature(C)'] + 50)

    # 원본 wind direction 제거
    df = df.drop(columns=["wind direction"])

    return df

In [32]:
X = apply_feature_engineering(X)
X_test = apply_feature_engineering(X_test)

print("최종 feature 개수:", len(X.columns))
X.head()

최종 feature 개수: 20


,temperature(C),wind speed(m/s),humidity(%),vapor pressure(hPa),dew point(C),atmospheric pressure(hPa),Sea level pressure(hPa),insolation time(hr),insolation(MJ/m2),cloud,temp_dew_diff,pressure_diff,wind_dir_sin,wind_dir_cos,wind_u,wind_v,apparent_temp,THI,humidity_temp_ratio,pressure_temp_ratio
0,23.1,1.3,84.0,23.7,20.2,999.3,1005.5,NaN,NaN,10,2.9,6.2,1.000000,6.123234e-17,7.960204e-17,1.300000,17.01,72.20896,1.149111,13.670315
1,25.5,1.1,63.0,20.5,17.9,1017.3,1018.8,0.9,NaN,4,7.6,1.5,0.939693,3.420201e-01,3.762222e-01,1.033662,18.48,73.85035,0.834437,13.474172
2,4.2,2.1,88.0,7.2,2.3,1002.9,1030.4,NaN,NaN,8,1.9,27.5,-0.939693,-3.420201e-01,-7.182423e-01,-1.973355,3.82,40.77704,1.623616,18.503690
3,5.4,1.0,21.0,1.9,-15.2,1027.4,1030.8,1.0,NaN,6,20.6,3.4,1.000000,6.123234e-17,6.123234e-17,1.000000,3.99,48.79366,0.379061,18.545126
4,23.6,2.7,26.0,7.5,2.9,1004.1,1014.4,1.0,3.39,0,20.7,10.3,-0.939693,-3.420201e-01,-9.234544e-01,-2.537170,16.78,67.77264,0.353261,13.642663


In [33]:
# XGBoost
xgb_model = XGBClassifier(
    n_estimators=800,
    learning_rate=0.015,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.5,
    random_state=42,
    n_jobs=-1
)

# HistGradientBoosting → sklearn
hgb_model = HistGradientBoostingClassifier(
    max_iter=800,
    learning_rate=0.015,
    max_depth=7,
    min_samples_leaf=20,
    l2_regularization=0.5,
    random_state=42
)

# CatBoost → 범주형 + 결측치 처리
cat_model = CatBoostClassifier(
    iterations=800,
    learning_rate=0.02,
    depth=6,
    l2_leaf_reg=3,
    verbose=False,
    random_state=42
)

In [34]:
# 앙상블
# 모델 리스트
estimators = [
    ('xgb', xgb_model),
    ('hgb', hgb_model),
    ('cat', cat_model)
]

# Soft Voting → 확률 평균
# CatBoost를 조금 더 신뢰해서 weight 높임
voting_clf = VotingClassifier(
    estimators=estimators,
    voting='soft',
    weights=[1, 1, 1.2]
)

In [35]:
print("5-Fold 시작")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds_probs = np.zeros((len(X_test), len(np.unique(y))))

fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"Fold {fold+1}")

    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

    # 학습
    voting_clf.fit(X_tr, y_tr)

    # 검증
    val_pred = voting_clf.predict(X_va)
    oof_preds[val_idx] = val_pred

    score = accuracy_score(y_va, val_pred)
    fold_scores.append(score)

    print("Accuracy:", score)

    # 테스트 예측 확률 누적
    test_preds_probs += voting_clf.predict_proba(X_test) / skf.n_splits

5-Fold 시작
Fold 1
Accuracy: 0.8228155339805825
Fold 2
Accuracy: 0.7815533980582524
Fold 3
Accuracy: 0.75
Fold 4
Accuracy: 0.781021897810219
Fold 5
Accuracy: 0.7664233576642335


In [36]:
print("-" * 30)

oof_accuracy = accuracy_score(y, oof_preds)
print(f"OOF Accuracy: {oof_accuracy:.4f} ({oof_accuracy*100:.2f}%)")

print("-" * 30)

# 최종 예측
final_test_preds = np.argmax(test_preds_probs, axis=1)

# 제출 파일 생성
submission = pd.DataFrame({
    'weather label': final_test_preds
})

submission.to_csv('pred_advanced.csv', index=False)


------------------------------
OOF Accuracy: 0.7804 (78.04%)
------------------------------
